In [160]:
import pandas as pd
import json
import os
import deepsig
from IPython.display import display

In [161]:
cols = ['dataset', 'method', 'fitness_rule', 'fitness', 'ACC', 'MCC', 'f1_score', 'avg_odds_diff', 'stat_par_diff', 'eq_opp_diff']

In [162]:
def read_csv_files_from_folder(folder_path):
    dfs = []
    for file_name in os.listdir(folder_path):
        if file_name.endswith(".csv"):
            file_path = os.path.join(folder_path, file_name)
            dfs.append(pd.read_csv(file_path))
    return pd.concat(dfs, ignore_index=True)

mlp_baseline_results = pd.read_csv('simple_mlp_results.csv')
mlp_baseline_results.replace({'simple_mlp_initializer': 'MLP'}, inplace=True)
mlp_xi_reg_results = pd.read_csv('mlp_xi_reg_results.csv')
mlp_xi_reg_results.replace({'mlp_xi_reg_initializer': r'MLP+SDR$_{\xi}$'}, inplace=True)

# MLP variantes do tradeoff_results
mlp_standard_l2_results = pd.read_csv('../tradeoff_results/mlp_standard_l2_results.csv')
mlp_standard_l2_results.replace({'mlp_standard_l2_initializer': r'MLP+L2'}, inplace=True)
mlp_featurewise_l2_results = pd.read_csv('../tradeoff_results/mlp_featurewise_l2_results.csv')
mlp_featurewise_l2_results.replace({'mlp_featurewise_l2_initializer': r'MLP+L2^{(1)}'}, inplace=True)
mlp_preg_results = pd.read_csv('../tradeoff_results/mlp_preg_results.csv')
mlp_preg_results.replace({'mlp_preg_initializer': r'MLP+SDR$_{\rho}$'}, inplace=True)
mlp_sreg_results = pd.read_csv('../tradeoff_results/mlp_sreg_results.csv')
mlp_sreg_results.replace({'mlp_sreg_initializer': r'MLP+SDR$_{\rho_s}$'}, inplace=True)
mlp_kreg_results = pd.read_csv('../tradeoff_results/mlp_kreg_results.csv')
mlp_kreg_results.replace({'mlp_kreg_initializer': r'MLP+SDR$_{\tau}$'}, inplace=True)

mlp_results = pd.concat([
    mlp_baseline_results, mlp_xi_reg_results,
    mlp_standard_l2_results, mlp_featurewise_l2_results,
    mlp_preg_results, mlp_sreg_results, mlp_kreg_results
])

ftl_baseline_results = pd.read_csv('ftl_mlp_results.csv')
ftl_baseline_results.replace({'ftl_mlp_initializer': 'FTL'}, inplace=True)
ftl_xi_reg_results = pd.read_csv('ftl_mlp_xi_reg_results.csv')
ftl_xi_reg_results.replace({'ftl_mlp_xi_reg_initializer': r'FTL+SDR$_{\xi}$'}, inplace=True)
ftl_results = pd.concat([ftl_baseline_results, ftl_xi_reg_results])

# HIFI results are stored as raw per-run CSVs produced by ablation.py's hifi_initializer
hifi_result = pd.read_csv('hifi_results.csv')
hifi_result.replace({'hifi_initializer': 'HIFI'}, inplace=True)

hifi_results = pd.concat([hifi_result, ftl_xi_reg_results])

full_results = pd.concat([mlp_results, ftl_results, hifi_results])


In [163]:
for results in [mlp_results,ftl_results,hifi_results, full_results]:
    results.replace({'adult_dataset_reader': 'Adult Income', 'compas_dataset_reader': 'Compas Recidivism', 'german_dataset_reader': 'German Credit', 'bank_dataset_reader': 'Bank Marketing'}, inplace=True)
    results.rename(columns={'avg_odds_diff': 'Equalized Odds', 'stat_par_diff': 'Statistical Parity', 'eq_opp_diff': 'Equal Opportunity', 'MCC': 'Mathew Correlation', 'ACC': 'Accuracy'}, inplace=True)

In [164]:
fitness_rules_target_metrics = {
    'mcc_parity': {'performance': 'Mathew Correlation', 'fairness': 'Statistical Parity'},
    'mcc_opportunity': {'performance': 'Mathew Correlation', 'fairness': 'Equal Opportunity'},
    'mcc_odds': {'performance': 'Mathew Correlation', 'fairness': 'Equalized Odds'},
    'acc_parity': {'performance': 'Accuracy', 'fairness': 'Statistical Parity'},
    'acc_opportunity': {'performance': 'Accuracy', 'fairness': 'Equal Opportunity'},
    'acc_odds': {'performance': 'Accuracy', 'fairness': 'Equalized Odds'}
}

fitness_rules_target_metrics = {
    'mcc_parity': ('Mathew Correlation', 'Statistical Parity'),
    'mcc_opportunity': ('Mathew Correlation', 'Equal Opportunity'),
    'mcc_odds': ('Mathew Correlation', 'Equalized Odds'),
    'acc_parity': ('Accuracy', 'Statistical Parity'),
    'acc_opportunity': ('Accuracy', 'Equal Opportunity'),
    'acc_odds': ('Accuracy', 'Equalized Odds')
}
fitness_rules_abvr = {
    'mcc_parity': 'Max(MCC - Stat. Parity)',
    'mcc_opportunity': 'Max(MCC - Eq. Odds)',
    'mcc_odds': 'Max(MCC - Eq. Opp.)',
    'acc_parity': 'Max(Acc - Stat. Parity)',
    'acc_opportunity': 'Max(Acc - Eq. Odds)',
    'acc_odds': 'Max(Acc - Eq. Opp.)'
}

for results in [mlp_results,ftl_results,hifi_results,full_results]:
    results['Performance'] = 0
    results['Fairness'] = 0
    results['Fitness Rule'] = ''
    for fitness_rule, (performance_metric, fairness_metric) in fitness_rules_target_metrics.items():
        results.loc[results.fitness_rule == fitness_rule,'Performance'] = results.loc[results.fitness_rule == fitness_rule,performance_metric]
        results.loc[results.fitness_rule == fitness_rule,'Fairness'] = results.loc[results.fitness_rule == fitness_rule,fairness_metric]
        results.loc[results.fitness_rule == fitness_rule,'Fitness Rule Abvr'] = fitness_rules_abvr[fitness_rule]
        results.loc[results.fitness_rule == fitness_rule,'Fitness Rule'] = 'Max(%s - %s)' % fitness_rules_target_metrics[fitness_rule]

/var/folders/z5/rq0dv5jj45qccc_tc39171cm0000gn/T/ipykernel_98211/2887299846.py:32: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '[0.58517004 0.58512823 0.5840247  0.5836371  0.57625737 0.5769683
 0.58075151 0.57811729 0.57147862 0.55847176 0.56402881 0.58191431
 0.58343253 0.57740222 0.51579307 0.30958237 0.37962947 0.55899189
 0.53558872 0.2598879  0.57717035 0.50086739 0.33275221 0.37929293
 0.47729109 0.29196869 0.29037704 0.52303609 0.27480633 0.38668781
 0.50626951 0.29960979 0.4042848  0.52119218 0.2936422  0.33522388
 0.49730721 0.25112662 0.35119067 0.510052   0.26843855 0.36081008
 0.52157495 0.30114722 0.42995147 0.48267704 0.29056691 0.23139881
 0.54223416 0.23653437 0.27041017 0.52121263 0.25759348 0.23912165
 0.5223892  0.27406652 0.22303564 0.54094974 0.27699558 0.30939251
 0.51731345 0.30339828 0.37825089 0.52113734 0.26323857 0.33963196
 0.5538225  0.27850228 0.30234984 0.51877126 0.29494101 0.26489

In [165]:
datasets = ['Adult Income', 'Bank Marketing', 'Compas Recidivism','German Credit']
datasets

['Adult Income', 'Bank Marketing', 'Compas Recidivism', 'German Credit']

In [166]:
fitness_rules = ['mcc_parity', 'mcc_opportunity', 'mcc_odds', 'acc_parity', 'acc_opportunity', 'acc_odds']
fitness_rules

['mcc_parity',
 'mcc_opportunity',
 'mcc_odds',
 'acc_parity',
 'acc_opportunity',
 'acc_odds']

In [167]:
ftl_methods = ['FTL', r'FTL+SDR$_{\xi}$']
mlp_methods = ['MLP', r'MLP+SDR$_{\xi}$']
hifi_methods = ['HIFI', r'FTL+SDR$_{\xi}$']
significances = []
grouped_results_list = []

In [168]:
for path, methods, results in zip(['mlp_multi_aso_data_list.json', 'ftl_multi_aso_data_list.json', 'hifi_multi_aso_data_list.json'],
                                  [mlp_methods, ftl_methods, hifi_methods],
                                  (mlp_results, ftl_results, hifi_results)):
    method = methods[0]
    if os.path.exists(path):
        with open(path) as file:
            multi_aso_data_list = json.load(file)
    else:    
        multi_aso_data_list = []
        for d in datasets:
            for f in fitness_rules:
                
                baseline = results.loc[ (results['dataset'] == d) &
                                         (results['fitness_rule'] == f) &
                                         (results['method'] == methods[0]) ]\
                                .fitness.tolist()
                crp = results.loc[ (results['dataset'] == d) &
                                         (results['fitness_rule'] == f) &
                                         (results['method'] == methods[1]) ]\
                                .fitness.tolist()

                print(d, f, methods[0], methods[1])
                print(len(baseline), len(crp))

                # Teste bidirecional
                min_eps_forward = deepsig.aso(crp, baseline, confidence_level=0.95)
                min_eps_backward = deepsig.aso(baseline, crp, confidence_level=0.95)
                
                # Classificação com os thresholds especificados
                if min_eps_forward < 0.2:
                    interpretation = "significantly_better"
                    min_eps = min_eps_forward
                elif min_eps_forward < 0.5:
                    interpretation = "better"
                    min_eps = min_eps_forward
                elif min_eps_backward < 0.2:
                    interpretation = "significantly_worse"
                    min_eps = min_eps_backward
                elif min_eps_backward < 0.5:
                    interpretation = "worse"
                    min_eps = min_eps_backward
                else:
                    interpretation = "tie"
                    min_eps = min_eps_forward
                
                multi_aso_data_list.append({
                    'fitness_rule': f, 
                    'dataset': d, 
                    'min_eps': min_eps,
                    'min_eps_forward': min_eps_forward,
                    'min_eps_backward': min_eps_backward,
                    'interpretation': interpretation
                })
        with open(path, 'w') as file:
            json.dump(multi_aso_data_list, file)

    significance = pd.DataFrame(multi_aso_data_list)
    
    # Tabela de valores epsilon (forward e backward)
    eps_table_data = []
    for fitness_rule in fitness_rules:
        row_data = {'fitness_rule': fitness_rule}
        for dataset in datasets:
            subset = significance[(significance['fitness_rule'] == fitness_rule) & 
                                 (significance['dataset'] == dataset)]
            if not subset.empty:
                row = subset.iloc[0]
                row_data[f'{dataset}_forward'] = f"{row['min_eps_forward']:.3f}"
                row_data[f'{dataset}_backward'] = f"{row['min_eps_backward']:.3f}"
            else:
                row_data[f'{dataset}_forward'] = '-'
                row_data[f'{dataset}_backward'] = '-'
        eps_table_data.append(row_data)
    
    eps_df = pd.DataFrame(eps_table_data)
    eps_df.set_index('fitness_rule', inplace=True)
    
    # Criar MultiIndex columns para tabela de epsilon
    eps_columns = []
    for dataset in datasets:
        eps_columns.extend([
            (dataset, 'Forward'),
            (dataset, 'Backward')
        ])
    
    eps_df.columns = pd.MultiIndex.from_tuples(eps_columns)
    
    # Salvar tabela de epsilon
    eps_df.to_latex(f'tables/aso_epsilon_{method.lower()}_crp.tex')
    print(f'\n{method} - ASO Epsilon Values (Forward/Backward)')
    display(eps_df)
    
    # Também criar a tabela simplificada original
    pivot_df = significance.pivot_table(index='fitness_rule', columns='dataset', values='min_eps').sort_values(by='fitness_rule', ascending=False)
    pivot_df.to_latex(f'tables/aso_results_{method.lower()}_crp.tex')
    print(f'\n{method} - Summary ASO Results')
    display(pivot_df)


MLP - ASO Epsilon Values (Forward/Backward)


Adult Income          Bank Marketing           \
                     Forward Backward        Forward Backward   
fitness_rule                                                    
mcc_parity             0.790    1.000          0.986    1.000   
mcc_opportunity        0.276    1.000          1.000    0.906   
mcc_odds               0.052    1.000          1.000    0.470   
acc_parity             0.732    1.000          1.000    0.209   
acc_opportunity        1.000    1.000          0.177    1.000   
acc_odds               0.305    1.000          1.000    0.369   

                Compas Recidivism          German Credit           
                          Forward Backward       Forward Backward  
fitness_rule                                                       
mcc_parity                  0.009    1.000         1.000    0.539  
mcc_opportunity             0.004    1.000         1.000    0.743  
mcc_odds                    0.198    1.000         1.000    0.628  
acc_parity                  0.418    1.000         0.870    1.000  
acc_opportunity             0.049    1.000         0.254    1.000  
acc_odds                    0.007    1.000         0.615    1.000


MLP - Summary ASO Results


dataset,Adult Income,Bank Marketing,Compas Recidivism,German Credit
fitness_rule,,,,
mcc_parity,0.789509,0.985749,0.009040,1.000000
mcc_opportunity,0.275891,1.000000,0.003632,1.000000
mcc_odds,0.051554,0.470019,0.198298,1.000000
acc_parity,0.731755,0.208639,0.418168,0.869970
acc_opportunity,1.000000,0.176787,0.049109,0.254203
acc_odds,0.304816,0.368617,0.006884,0.614694



FTL - ASO Epsilon Values (Forward/Backward)


...

dataset,Adult Income,Bank Marketing,Compas Recidivism,German Credit
fitness_rule,,,,
mcc_parity,0.316138,0.467764,1.000000,0.272724
mcc_opportunity,0.374905,0.260068,0.182578,0.086913
mcc_odds,0.317497,1.000000,0.557753,0.364560
acc_parity,0.433384,1.000000,0.057004,0.486228
acc_opportunity,0.587631,0.660108,0.398807,1.000000
acc_odds,0.438244,0.449773,0.204134,0.299867



HIFI - ASO Epsilon Values (Forward/Backward)


Adult Income          Bank Marketing           \
                     Forward Backward        Forward Backward   
fitness_rule                                                    
mcc_parity             0.000    0.997          0.000    0.998   
mcc_opportunity        0.017    1.000          0.912    1.000   
mcc_odds               0.022    1.000          0.027    1.000   
acc_parity             0.000    0.996          1.000    0.820   
acc_opportunity        0.043    1.000          1.000    0.531   
acc_odds               0.018    1.000          1.000    1.000   

                Compas Recidivism          German Credit           
                          Forward Backward       Forward Backward  
fitness_rule                                                       
mcc_parity                  0.000    0.998         0.198    1.000  
mcc_opportunity             0.022    1.000         1.000    0.511  
mcc_odds                    0.006    1.000         1.000    0.687  
acc_parity                  0.000    1.000         1.000    0.315  
acc_opportunity             0.027    1.000         0.857    1.000  
acc_odds                    0.000    0.998         1.000    0.134


HIFI - Summary ASO Results


dataset,Adult Income,Bank Marketing,Compas Recidivism,German Credit
fitness_rule,,,,
mcc_parity,0.000000,0.000000,1.223873e-07,0.197881
mcc_opportunity,0.017179,0.912036,2.242494e-02,1.000000
mcc_odds,0.022113,0.026805,5.854291e-03,1.000000
acc_parity,0.000000,1.000000,0.000000e+00,0.314754
acc_opportunity,0.042555,1.000000,2.724459e-02,0.857379
acc_odds,0.018341,1.000000,5.691066e-05,0.134259


In [169]:
# Criar tabela de interpretação comparando os 3 baselines
# Carregar todos os dados ASO
all_aso_data = {}
baseline_names = ['MLP', 'FTL', 'HIFI']

# Mapeamento de interpretação para símbolos
symbol_map = {
    'significantly_better': '++',
    'better': '+',
    'tie': '≈',
    'worse': '-',
    'significantly_worse': '--'
}

for path, baseline in zip(['mlp_multi_aso_data_list.json', 'ftl_multi_aso_data_list.json', 'hifi_multi_aso_data_list.json'],
                          baseline_names):
    with open(path) as file:
        all_aso_data[baseline] = pd.DataFrame(json.load(file))

# Criar tabela de interpretação com subcolunas por dataset
interp_table_data = []
for fitness_rule in fitness_rules:
    row_data = {'fitness_rule': fitness_rule}
    for dataset in datasets:
        for baseline in baseline_names:
            subset = all_aso_data[baseline][(all_aso_data[baseline]['fitness_rule'] == fitness_rule) & 
                                            (all_aso_data[baseline]['dataset'] == dataset)]
            if not subset.empty:
                row = subset.iloc[0]
                # Converter interpretação para símbolo
                row_data[f'{dataset}_{baseline}'] = symbol_map.get(row['interpretation'], '?')
            else:
                row_data[f'{dataset}_{baseline}'] = '-'
    interp_table_data.append(row_data)

interp_df = pd.DataFrame(interp_table_data)
interp_df.set_index('fitness_rule', inplace=True)

# Criar MultiIndex columns para tabela de interpretação
interp_columns = []
for dataset in datasets:
    for baseline in baseline_names:
        interp_columns.append((dataset, baseline))

interp_df.columns = pd.MultiIndex.from_tuples(interp_columns)

# Salvar tabela de interpretação
interp_df.to_latex('tables/aso_interpretation_comparison.tex')
print('\nASO Interpretation Comparison (All Baselines)')
display(interp_df)

# Criar tabela resumo com contagem de interpretações
summary_data = []
for baseline in baseline_names:
    counts = all_aso_data[baseline]['interpretation'].value_counts()
    summary_row = {'Baseline': baseline}
    for interp_type in ['significantly_better', 'better', 'tie', 'worse', 'significantly_worse']:
        summary_row[interp_type] = counts.get(interp_type, 0)
    summary_data.append(summary_row)

summary_df = pd.DataFrame(summary_data)
summary_df.set_index('Baseline', inplace=True)

# Renomear colunas para LaTeX
summary_df.columns = ['Sig. Better (++)', 'Better (+)', 'Tie (≈)', 'Worse (-)', 'Sig. Worse (--)']

# Adicionar linha de somatório
total_row = summary_df.sum()
total_row.name = 'Total'
summary_df = pd.concat([summary_df, total_row.to_frame().T])

# Salvar tabela resumo
summary_df.to_latex('tables/aso_interpretation_summary.tex')
print('\nASO Interpretation Summary (Global Counts)')
display(summary_df)


ASO Interpretation Comparison (All Baselines)


Adult Income          Bank Marketing           \
                         MLP FTL HIFI            MLP FTL HIFI   
fitness_rule                                                    
mcc_parity                 ≈   +   ++              ≈   -   ++   
mcc_opportunity            +   +   ++              ≈   -    ≈   
mcc_odds                  ++   +   ++              -   ≈   ++   
acc_parity                 ≈   +   ++              -   ≈    ≈   
acc_opportunity            ≈   ≈   ++             ++   ≈    ≈   
acc_odds                   +   +   ++              -   -    ≈   

                Compas Recidivism          German Credit           
                              MLP FTL HIFI           MLP FTL HIFI  
fitness_rule                                                       
mcc_parity                     ++   ≈   ++             ≈   +   ++  
mcc_opportunity                ++  ++   ++             ≈  --    ≈  
mcc_odds                       ++   ≈   ++             ≈   -    ≈  
acc_parity                      +  ++   ++             ≈   -    -  
acc_opportunity                ++   -   ++             +   ≈    ≈  
acc_odds                       ++   +   ++             ≈   -   --


ASO Interpretation Summary (Global Counts)


,Sig. Better (++),Better (+),Tie (≈),Worse (-),Sig. Worse (--)
MLP,7,4,10,3,0
FTL,2,7,7,7,1
HIFI,15,0,7,1,1
Total,24,11,24,11,2


In [170]:
grouped_results = full_results\
    .groupby(['fitness_rule', 'dataset', 'method'])\
    .agg({'fitness': ['mean', 'std', 'count'], 'Performance': ['mean', 'std'], 'Fairness': ['mean', 'std']})\
    .sort_values(by=['fitness_rule', 'dataset', ('fitness','mean')], ascending=[False, True, False])
grouped_results['formatted_fitness'] = grouped_results.apply(lambda row: f"${row[('fitness', 'mean')]:.3f} \pm{row[('fitness', 'std')]:.2f}$", axis=1)
grouped_results['formatted_performance'] = grouped_results.apply(lambda row: f"${row[('Performance', 'mean')]:.3f} \pm{row[('Performance', 'std')]:.2f}$", axis=1)
grouped_results['formatted_fairness'] = grouped_results.apply(lambda row: f"${row[('Fairness', 'mean')]:.3f} \pm{row[('Fairness', 'std')]:.2f}$", axis=1)
display(grouped_results)

fitness                  \
                                                 mean       std count   
fitness_rule dataset       method                                       
mcc_parity   Adult Income  FTL+SDR$_{\xi}$   0.494373  0.014363    30   
                           FTL               0.486772  0.017670    25   
                           MLP+L2^{(1)}      0.396227  0.013025    15   
                           HIFI              0.395732  0.014024    15   
                           MLP+SDR$_{\rho}$  0.393134  0.012324    18   
...                                               ...       ...   ...   
acc_odds     German Credit MLP+SDR$_{\rho}$  0.641057  0.038716    16   
                           MLP+SDR$_{\xi}$   0.640351  0.062612    15   
                           MLP+L2^{(1)}      0.639753  0.041551    15   
                           FTL+SDR$_{\xi}$   0.630951  0.062479    30   
                           MLP               0.619366  0.070376    30   

                                            Performance            Fairness  \
                                                   mean       std      mean   
fitness_rule dataset       method                                             
mcc_parity   Adult Income  FTL+SDR$_{\xi}$     0.516994  0.017254  0.022621   
                           FTL                 0.508716  0.021961  0.021944   
                           MLP+L2^{(1)}        0.578930  0.009622  0.182703   
                           HIFI                0.572667  0.009679  0.176935   
                           MLP+SDR$_{\rho}$    0.580085  0.010404  0.186951   
...                                                 ...       ...       ...   
acc_odds     German Credit MLP+SDR$_{\rho}$    0.734375  0.028976  0.093318   
                           MLP+SDR$_{\xi}$     0.747667  0.020430  0.107316   
                           MLP+L2^{(1)}        0.739667  0.022398  0.099914   
                           FTL+SDR$_{\xi}$     0.720667  0.028337  0.089716   
                           MLP                 0.740000  0.031786  0.120634   

                                                      formatted_fitness  \
                                                  std                     
fitness_rule dataset       method                                         
mcc_parity   Adult Income  FTL+SDR$_{\xi}$   0.021924   $0.494 \pm0.01$   
                           FTL               0.019962   $0.487 \pm0.02$   
                           MLP+L2^{(1)}      0.011855   $0.396 \pm0.01$   
                           HIFI              0.014412   $0.396 \pm0.01$   
                           MLP+SDR$_{\rho}$  0.013465   $0.393 \pm0.01$   
...                                               ...               ...   
acc_odds     German Credit MLP+SDR$_{\rho}$  0.044720   $0.641 \pm0.04$   
                           MLP+SDR$_{\xi}$   0.058156   $0.640 \pm0.06$   
                           MLP+L2^{(1)}      0.045268   $0.640 \pm0.04$   
                           FTL+SDR$_{\xi}$   0.073788   $0.631 \pm0.06$   
                           MLP               0.068075   $0.619 \pm0.07$   

                                            formatted_performance  \
                                                                    
fitness_rule dataset       method                                   
mcc_parity   Adult Income  FTL+SDR$_{\xi}$        $0.517 \pm0.02$   
                           FTL                    $0.509 \pm0.02$   
                           MLP+L2^{(1)}           $0.579 \pm0.01$   
                           HIFI                   $0.573 \pm0.01$   
                           MLP+SDR$_{\rho}$       $0.580 \pm0.01$   
...                                                           ...   
acc_odds     German Credit MLP+SDR$_{\rho}$       $0.734 \pm0.03$   
                           MLP+SDR$_{\xi}$        $0.748 \pm0.02$   
                           MLP+L2^{(1)}           $0.740 \pm0.02$   
                           FTL+SDR$_{

In [171]:
# Adiciona colunas numericas de ASO e ASO reverso ao grouped_results
# ASO = forward (baseline -> metodo), ASO reverso = backward (metodo -> baseline)
# MLP variantes sao comparadas com MLP+SDR_xi (fonte: tradeoff mlp_xi_multi_aso_results.json)
# FTL e HIFI sao comparados com FTL+SDR_xi (fonte: all_aso_data)

# Carrega ASO do tradeoff_results (MLP+SDR_xi vs outras variantes MLP)
tradeoff_mlp_aso_path = '../tradeoff_results/mlp_xi_multi_aso_results.json'
with open(tradeoff_mlp_aso_path) as file:
    tradeoff_mlp_aso_list = json.load(file)

mlp_xi_aso_rows = []
for entry in tradeoff_mlp_aso_list:
    for method, comp_data in entry['comparisons'].items():
        mlp_xi_aso_rows.append({
            'fitness_rule': entry['fitness_rule'],
            'dataset': entry['dataset'],
            'method': method,
            'min_eps_forward': comp_data['eps_forward'],
            'min_eps_backward': comp_data['eps_backward']
        })
mlp_xi_aso_df = pd.DataFrame(mlp_xi_aso_rows)

# Mapeamento: nome do metodo -> fonte de ASO e filtro
# Formato: (df, filter_col_name, filter_value)
# Para FTL/HIFI as fontes all_aso_data nao tem coluna 'method', entao filter_col_name=None
mlp_variants = ['MLP', r'MLP+L2', r'MLP+L2^{(1)}',
                r'MLP+SDR$_{\rho}$', r'MLP+SDR$_{\rho_s}$', r'MLP+SDR$_{\tau}$']

aso_method_sources = {}
for m in mlp_variants:
    aso_method_sources[m] = (mlp_xi_aso_df, 'method', m)
aso_method_sources['FTL'] = (all_aso_data['FTL'], None, None)
aso_method_sources['HIFI'] = (all_aso_data['HIFI'], None, None)
# MLP+SDR_xi e FTL+SDR_xi sao baselines; nao tem ASO nessa comparacao

aso_vals = []
raso_vals = []

for fitness_rule, dataset, method in grouped_results.index:
    source = aso_method_sources.get(method)
    if source is not None:
        df, filter_col, filter_val = source
        mask = (df['fitness_rule'] == fitness_rule) & (df['dataset'] == dataset)
        if filter_col is not None:
            mask = mask & (df[filter_col] == filter_val)
        row = df[mask]
        if not row.empty:
            aso_vals.append(row.iloc[0]['min_eps_forward'])
            raso_vals.append(row.iloc[0]['min_eps_backward'])
        else:
            aso_vals.append(float('nan'))
            raso_vals.append(float('nan'))
    else:
        aso_vals.append(float('nan'))
        raso_vals.append(float('nan'))

grouped_results['ASO'] = aso_vals
grouped_results['ASO reverso'] = raso_vals

display(grouped_results)


fitness                  \
                                                 mean       std count   
fitness_rule dataset       method                                       
mcc_parity   Adult Income  FTL+SDR$_{\xi}$   0.494373  0.014363    30   
                           FTL               0.486772  0.017670    25   
                           MLP+L2^{(1)}      0.396227  0.013025    15   
                           HIFI              0.395732  0.014024    15   
                           MLP+SDR$_{\rho}$  0.393134  0.012324    18   
...                                               ...       ...   ...   
acc_odds     German Credit MLP+SDR$_{\rho}$  0.641057  0.038716    16   
                           MLP+SDR$_{\xi}$   0.640351  0.062612    15   
                           MLP+L2^{(1)}      0.639753  0.041551    15   
                           FTL+SDR$_{\xi}$   0.630951  0.062479    30   
                           MLP               0.619366  0.070376    30   

                                            Performance            Fairness  \
                                                   mean       std      mean   
fitness_rule dataset       method                                             
mcc_parity   Adult Income  FTL+SDR$_{\xi}$     0.516994  0.017254  0.022621   
                           FTL                 0.508716  0.021961  0.021944   
                           MLP+L2^{(1)}        0.578930  0.009622  0.182703   
                           HIFI                0.572667  0.009679  0.176935   
                           MLP+SDR$_{\rho}$    0.580085  0.010404  0.186951   
...                                                 ...       ...       ...   
acc_odds     German Credit MLP+SDR$_{\rho}$    0.734375  0.028976  0.093318   
                           MLP+SDR$_{\xi}$     0.747667  0.020430  0.107316   
                           MLP+L2^{(1)}        0.739667  0.022398  0.099914   
                           FTL+SDR$_{\xi}$     0.720667  0.028337  0.089716   
                           MLP                 0.740000  0.031786  0.120634   

                                                      formatted_fitness  \
                                                  std                     
fitness_rule dataset       method                                         
mcc_parity   Adult Income  FTL+SDR$_{\xi}$   0.021924   $0.494 \pm0.01$   
                           FTL               0.019962   $0.487 \pm0.02$   
                           MLP+L2^{(1)}      0.011855   $0.396 \pm0.01$   
                           HIFI              0.014412   $0.396 \pm0.01$   
                           MLP+SDR$_{\rho}$  0.013465   $0.393 \pm0.01$   
...                                               ...               ...   
acc_odds     German Credit MLP+SDR$_{\rho}$  0.044720   $0.641 \pm0.04$   
                           MLP+SDR$_{\xi}$   0.058156   $0.640 \pm0.06$   
                           MLP+L2^{(1)}      0.045268   $0.640 \pm0.04$   
                           FTL+SDR$_{\xi}$   0.073788   $0.631 \pm0.06$   
                           MLP               0.068075   $0.619 \pm0.07$   

                                            formatted_performance  \
                                                                    
fitness_rule dataset       method                                   
mcc_parity   Adult Income  FTL+SDR$_{\xi}$        $0.517 \pm0.02$   
                           FTL                    $0.509 \pm0.02$   
                           MLP+L2^{(1)}           $0.579 \pm0.01$   
                           HIFI                   $0.573 \pm0.01$   
                           MLP+SDR$_{\rho}$       $0.580 \pm0.01$   
...                                                           ...   
acc_odds     German Credit MLP+SDR$_{\rho}$       $0.734 \pm0.03$   
                           MLP+SDR$_{\xi}$        $0.748 \pm0.02$   
                           MLP+L2^{(1)}           $0.740 \pm0.02$   
                           FTL+SDR$_{

In [172]:
selected_columns = ['formatted_fitness', 'formatted_performance', 'formatted_fairness']

for fitness_rule in fitness_rules:
    grouped_results.loc[fitness_rule][selected_columns].to_latex(f'tables/grouped_results_{fitness_rule}_crp.tex')
     #.to_latex(f'tables/grouped_results_{fitness_rule}_crp.tex', columns=selected_columns))

In [173]:
# Resumos de interpretação ASO agregados por fitness_rule e por dataset

# Concatena todos os dados ASO em um único DataFrame
all_aso_df = pd.concat([v.assign(Baseline=k) for k, v in all_aso_data.items()])

# Ordem das interpretações e rótulos
interpretation_order = ['significantly_better', 'better', 'tie', 'worse', 'significantly_worse']
interpretation_labels = ['Sig. Better (++)', 'Better (+)', 'Tie (≈)', 'Worse (-)', 'Sig. Worse (--)']

# Agregado por fitness_rule
fitness_summary = (
    all_aso_df.groupby('fitness_rule')['interpretation']
    .value_counts()
    .unstack(fill_value=0)
)
fitness_summary = fitness_summary.reindex(columns=interpretation_order, fill_value=0)
fitness_summary.columns = interpretation_labels
fitness_summary.index.name = 'Fitness Rule'
fitness_summary = fitness_summary.rename(index=fitness_rules_abvr)
fitness_summary.loc['Total'] = fitness_summary.sum()
fitness_summary.to_latex('tables/aso_interpretation_summary_fitness_rule.tex')
print('ASO Interpretation Summary (Aggregated by Fitness Rule)')
display(fitness_summary)

# Agregado por dataset
dataset_summary = (
    all_aso_df.groupby('dataset')['interpretation']
    .value_counts()
    .unstack(fill_value=0)
)
dataset_summary = dataset_summary.reindex(columns=interpretation_order, fill_value=0)
dataset_summary.columns = interpretation_labels
dataset_summary.index.name = 'Dataset'
dataset_summary.loc['Total'] = dataset_summary.sum()
dataset_summary.to_latex('tables/aso_interpretation_summary_dataset.tex')
print('ASO Interpretation Summary (Aggregated by Dataset)')
display(dataset_summary)


ASO Interpretation Summary (Aggregated by Fitness Rule)


,Sig. Better (++),Better (+),Tie (≈),Worse (-),Sig. Worse (--)
Fitness Rule,,,,,
Max(Acc - Eq. Opp.),3,3,2,3,1
Max(Acc - Eq. Odds),4,1,6,1,0
Max(Acc - Stat. Parity),3,2,4,3,0
Max(MCC - Eq. Opp.),5,1,4,2,0
Max(MCC - Eq. Odds),4,2,4,1,1
Max(MCC - Stat. Parity),5,2,4,1,0
Total,24,11,24,11,2


ASO Interpretation Summary (Aggregated by Dataset)


,Sig. Better (++),Better (+),Tie (≈),Worse (-),Sig. Worse (--)
Dataset,,,,,
Adult Income,7,7,4,0,0
Bank Marketing,3,0,9,6,0
Compas Recidivism,13,2,2,1,0
German Credit,1,2,9,4,2
Total,24,11,24,11,2


In [174]:
# Exporta tabela montada com uma linha por metodo
# Colunas: fitness formatada, performance formatada, fairness formatada, ASO, ASO reverso
s = chr(92)
nl = chr(10)

dataset_short = {
    'Adult Income': 'Adult',
    'Bank Marketing': 'Bank',
    'Compas Recidivism': 'COMPAS',
    'German Credit': 'German'
}

def method_latex(m):
    return f'${s}mathrm{{{m.replace(chr(36), "")}}}$'

def fmt_aso(v):
    return f'{v:.3f}' if not pd.isna(v) else '-'

# Colunas do grouped_results estao em MultiIndex; achata para acesso escalar
gr = grouped_results.copy()
gr.columns = [
    c[0] if (isinstance(c, tuple) and (pd.isna(c[1]) or c[1] == '')) else ' '.join(c)
    for c in gr.columns
]

lines = [
    s + 'begin{table}[htbp]',
    s + 'centering',
    s + 'caption{Fitness, performance, fairness, ASO and reverse ASO for all methods and optimization scenarios.}',
    s + 'label{tab:perf_fair_aso_summary}',
    s + 'begin{tabular}{lllccccc}',
    s + 'toprule',
    'Fitness Rule & Dataset & Method & Fitness & Performance & Fairness & ASO & ASO Reverse ' + s + s,
    s + 'midrule'
]

first = True
for fitness_rule in fitness_rules:
    if not first:
        lines.append(s + 'midrule')
    first = False
    for dataset in datasets:
        block = gr.loc[(fitness_rule, dataset)]
        for method, row in block.iterrows():
            cells = [
                fitness_rules_abvr[fitness_rule],
                dataset_short[dataset],
                method_latex(method),
                row['formatted_fitness'],
                row['formatted_performance'],
                row['formatted_fairness'],
                fmt_aso(row['ASO']),
                fmt_aso(row['ASO reverso'])
            ]
            lines.append(' & '.join(cells) + ' ' + s + s)

lines.extend([s + 'bottomrule', s + 'end{tabular}', s + 'end{table}'])
with open('tables/performance_fairness_summary.tex', 'w') as f:
    f.write(nl.join(lines))

print('Tabela salva em tables/performance_fairness_summary.tex')


Tabela salva em tables/performance_fairness_summary.tex


/var/folders/z5/rq0dv5jj45qccc_tc39171cm0000gn/T/ipykernel_98211/2211137864.py:43: PerformanceWarning: indexing past lexsort depth may impact performance.
  block = gr.loc[(fitness_rule, dataset)]
